In [53]:
import re
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from collections import Counter

In [ ]:
WINDOW = 20
MAX_VOCAB = 10000
EMBED_DIM = 96
LSTM_UNITS = 256
DROPOUT = 0.4
BATCH_SIZE = 128
EPOCHS = 30
VAL_RATIO = 0.10
SEED = 42
PATIENCE = 5 

np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Load and clean text

In [ ]:
with open("data.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = re.sub(r"[^a-z0-9!.?']+", " ", text.lower())
text = re.sub(r"\s+", " ", text).strip()

sentences = re.split(r"[.!?]", text)
sentences = [s.strip() for s in sentences if s.strip()]
print(f"Sentences: {len(sentences):,}")

Sentences: 7,517


## 2. Custom Tokenizer (replaces keras.preprocessing.text.Tokenizer)

In [56]:
class SimpleTokenizer:
    def __init__(self, num_words=None, oov_token="<OOV>"):
        self.num_words = num_words
        self.oov_token = oov_token
        self.word_index = {}
        self.index_word = {}

    def fit_on_texts(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(t.split())

        vocab_size = self.num_words - 1 if self.num_words else len(counter)
        most_common = counter.most_common(vocab_size)

        self.word_index = {self.oov_token: 1}
        for i, (word, _) in enumerate(most_common, start=2):
            self.word_index[word] = i

        self.index_word = {idx: word for word, idx in self.word_index.items()}

    def texts_to_sequences(self, texts):
        sequences = []
        oov_id = self.word_index.get(self.oov_token, 1)
        for t in texts:
            seq = [self.word_index.get(w, oov_id) for w in t.split()]
            sequences.append(seq)
        return sequences

    @property
    def word_index_size(self):
        return len(self.word_index) + 1 


tokenizer = SimpleTokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
tokenized_sentences = tokenizer.texts_to_sequences(sentences)

In [57]:
vocab_size

8177

## 3. Build sliding-window training pairs

In [58]:
X_list, y_list = [], []
for tokens in tokenized_sentences:
    if len(tokens) < 2:
        continue
    for target_index in range(1, len(tokens)):
        start_index = max(0, target_index - WINDOW)
        context = tokens[start_index:target_index]
        target = tokens[target_index]
        X_list.append(context)
        y_list.append(target)


def pad_sequences_pre(sequences, maxlen):
    padded = np.zeros((len(sequences), maxlen), dtype=np.int64)
    for i, seq in enumerate(sequences):
        trimmed = seq[-maxlen:]
        padded[i, maxlen - len(trimmed):] = trimmed
    return padded


X = pad_sequences_pre(X_list, WINDOW)
y = np.array(y_list, dtype=np.int64)

valid = y < vocab_size
X, y = X[valid], y[valid]

print(f"Vocabulary size: {vocab_size:,}")
print(f"Training examples: {len(X):,}")
print(f"X shape: {X.shape}  y shape: {y.shape}")

Vocabulary size: 8,177
Training examples: 101,731
X shape: (101731, 20)  y shape: (101731,)


## 4. Train/val split and DataLoaders

In [59]:
indices = np.random.permutation(len(X))
X, y = X[indices], y[indices]

split_index = int(len(X) * (1 - VAL_RATIO))
X_train, X_val = X[:split_index], X[split_index:]
y_train, y_val = y[:split_index], y[split_index:]


class NextWordDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).long()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = NextWordDataset(X_train, y_train)
val_ds = NextWordDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=(device.type == "cuda"))

## 5. Model: Embedding → LSTM → Dropout → LSTM → Dropout → Dense

In [60]:
class NextWordLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, lstm_units, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm1 = nn.LSTM(embed_dim, lstm_units, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(lstm_units, lstm_units, batch_first=True)
        self.dropout2 = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_units, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm1(x)
        x = self.dropout1(x)
        x, (h_n, _) = self.lstm2(x)
        x = self.dropout2(h_n[-1])  
        logits = self.fc(x)
        return logits


model = NextWordLSTM(vocab_size, EMBED_DIM, LSTM_UNITS, DROPOUT).to(device)
print(model)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

NextWordLSTM(
  (embedding): Embedding(8177, 96, padding_idx=0)
  (lstm1): LSTM(96, 256, batch_first=True)
  (dropout1): Dropout(p=0.4, inplace=False)
  (lstm2): LSTM(256, 256, batch_first=True)
  (dropout2): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=256, out_features=8177, bias=True)
)


In [61]:
def topk_correct(logits, yb, k=5):
    _, top_preds = logits.topk(k, dim=1)
    return (top_preds == yb.unsqueeze(1)).any(dim=1).sum().item()

## 6. Training loop with early stopping + checkpointing

In [62]:
best_val_loss = float("inf")
epochs_no_improve = 0
best_state_dict = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    train_correct_top5 = 0                                    # NEW

    for xb, yb in train_loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        train_correct += (logits.argmax(dim=1) == yb).sum().item()
        train_correct_top5 += topk_correct(logits, yb, k=5)   # NEW
        train_total += xb.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total
    train_acc_top5 = train_correct_top5 / train_total          # NEW

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    val_correct_top5 = 0                                        # NEW
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            logits = model(xb)
            loss = criterion(logits, yb)

            val_loss += loss.item() * xb.size(0)
            val_correct += (logits.argmax(dim=1) == yb).sum().item()
            val_correct_top5 += topk_correct(logits, yb, k=5)  # NEW
            val_total += xb.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_acc_top5 = val_correct_top5 / val_total                 # NEW

    print(f"Epoch {epoch}/{EPOCHS} - "
          f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} (top5: {train_acc_top5:.4f}) - "
          f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f} (top5: {val_acc_top5:.4f})")   # CHANGED

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        torch.save(best_state_dict, "best_next_word_lstm.pt")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}")
            break

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

Epoch 1/30 - train_loss: 6.5150 train_acc: 0.0672 (top5: 0.1819) - val_loss: 6.1223 val_acc: 0.0924 (top5: 0.2311)
Epoch 2/30 - train_loss: 5.9313 train_acc: 0.1034 (top5: 0.2585) - val_loss: 5.8258 val_acc: 0.1174 (top5: 0.2749)
Epoch 3/30 - train_loss: 5.6539 train_acc: 0.1196 (top5: 0.2862) - val_loss: 5.6910 val_acc: 0.1293 (top5: 0.2922)
Epoch 4/30 - train_loss: 5.4439 train_acc: 0.1309 (top5: 0.3039) - val_loss: 5.6178 val_acc: 0.1359 (top5: 0.3048)
Epoch 5/30 - train_loss: 5.2555 train_acc: 0.1403 (top5: 0.3212) - val_loss: 5.5596 val_acc: 0.1426 (top5: 0.3152)
Epoch 6/30 - train_loss: 5.0809 train_acc: 0.1509 (top5: 0.3351) - val_loss: 5.5313 val_acc: 0.1454 (top5: 0.3227)
Epoch 7/30 - train_loss: 4.9135 train_acc: 0.1578 (top5: 0.3476) - val_loss: 5.5453 val_acc: 0.1487 (top5: 0.3282)
Epoch 8/30 - train_loss: 4.7606 train_acc: 0.1636 (top5: 0.3600) - val_loss: 5.5487 val_acc: 0.1561 (top5: 0.3322)
Epoch 9/30 - train_loss: 4.6053 train_acc: 0.1721 (top5: 0.3720) - val_loss: 5.5

## 7. Save final model + tokenizer

In [63]:
torch.save(model.state_dict(), "next_word_lstm.pt")
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Saved: next_word_lstm.pt and tokenizer.pkl")

Saved: next_word_lstm.pt and tokenizer.pkl


## 8. Text generation (top-k sampling)

In [64]:
def generate_next_words(seed_text, num_words=20, top_k=10):
    model.eval()
    generated_text = seed_text.lower().strip()

    for _ in range(num_words):
        tokens = tokenizer.texts_to_sequences([generated_text])[0][-WINDOW:]
        if not tokens:
            break

        padded = pad_sequences_pre([tokens], WINDOW)
        x_tensor = torch.from_numpy(padded).long().to(device)

        with torch.no_grad():
            logits = model(x_tensor)
            probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]

        top_indices = np.argsort(probabilities)[-top_k:]
        top_probabilities = probabilities[top_indices]
        top_probabilities = top_probabilities / top_probabilities.sum()

        next_id = np.random.choice(top_indices, p=top_probabilities)
        next_word = tokenizer.index_word.get(next_id)

        if not next_word or next_word == "<OOV>":
            break

        generated_text += " " + next_word

    return generated_text


print(generate_next_words("once upon a time", num_words=20, top_k=10))

once upon a time to say it is a very thing that she was very possible to do and there is no doubt which
